# Statistics Lab Manual
## Descriptive Statistics using Python (Jupyter Notebook)

**Dataset used:** Forest Cover Type Dataset (REAL dataset, 581,012 rows, from `scikit-learn`)

**Level:** 3rd Semester — simple, beginner-friendly Python code

**Topics:**
1. Problem Statement
2. Problem Analysis Approach
3. Features of the Dataset
4. Load Dataset
5. Dependent and Independent Variables
6. Mean vs Median
7. Variance vs IQR
8. Skewness and Distribution Shape
9. Histogram, Boxplot, Density Plot
10. Pivot Tables

---


## 1. Problem Statement

A **forest research and land-management agency** working in the Roosevelt National
Forest, Colorado (USA) wants to study the **terrain conditions** (elevation, slope,
distance to water, distance to roads, sunlight exposure, etc.) of different patches of
forest, so that they can:

- Understand the typical **elevation** at which forest patches are recorded, and how
  spread out / consistent this elevation is.
- Check whether elevation and other terrain measurements are **normally distributed**
  or **skewed**, since skewed terrain data can bias land-management decisions and any
  future prediction models built on this data.
- Compare terrain conditions **across the 7 major forest cover types** (Spruce/Fir,
  Lodgepole Pine, Ponderosa Pine, Cottonwood/Willow, Aspen, Douglas-fir, Krummholz) to
  see which tree types grow at which elevations, slopes, and distances from
  water/roads.

**Business / research question:** *"What is the typical elevation of forest land in
this region, how much does it vary, and how does it differ across different forest
cover types?"*

To answer this, we will use a **real, large dataset** called the
**Forest Cover Type dataset**, collected by the US Forest Service using actual
cartographic (map-based) survey data — **no data is invented or simulated**.

## 2. Problem Analysis Approach

We will follow a standard **descriptive statistics / Exploratory Data Analysis (EDA)**
approach, step by step:

1. **Load the real dataset** into a pandas DataFrame and inspect its size and columns.
2. **Identify the dependent variable** (the terrain measurement we want to study —
   `Elevation`) and the **independent variables** (other terrain measurements and the
   forest cover type that might explain differences in elevation).
3. **Measure central tendency** — compute Mean and Median of Elevation to check if the
   data is centered symmetrically or pulled by extreme values.
4. **Measure spread** — compute Variance, Standard Deviation, and IQR to understand how
   spread out the elevation values are, and to detect outliers.
5. **Measure shape** — compute Skewness to formally describe whether the distribution
   leans left, right, or is symmetric.
6. **Visualize** the distribution using a Histogram, Boxplot, and Density plot so the
   shape can be seen, not just calculated.
7. **Group and summarize** using Pivot Tables — comparing average elevation (and other
   measures) across different forest cover types and wilderness areas.

This approach — load → identify variables → compute → visualize → summarize by group —
is the same general workflow used in almost any real-world data analysis project.

## 3. Features of the Dataset

The Forest Cover Type dataset has **581,012 rows** (one row per 30m x 30m patch of
forest land) and **54 columns** in total. The important ones we will use are:

| Feature | Description |
|---|---|
| Elevation | Elevation of the land in meters above sea level |
| Aspect | Compass direction the slope faces (degrees, 0-360) |
| Slope | Steepness of the slope (degrees) |
| Horizontal_Distance_To_Hydrology | Horizontal distance to nearest water source (m) |
| Vertical_Distance_To_Hydrology | Vertical distance to nearest water source (m) |
| Horizontal_Distance_To_Roadways | Horizontal distance to nearest road (m) |
| Hillshade_9am / Noon / 3pm | Amount of sunlight/shade on the land at different times of day (0-255) |
| Horizontal_Distance_To_Fire_Points | Distance to nearest wildfire ignition point (m) |
| Wilderness_Area | Which of 4 wilderness areas the land belongs to |
| Soil_Type | Which of 40 soil types is present |
| Cover_Type | The dominant tree species growing there (1 to 7) — used for grouping in pivot tables |

The **Cover_Type** codes stand for:
1 = Spruce/Fir, 2 = Lodgepole Pine, 3 = Ponderosa Pine, 4 = Cottonwood/Willow,
5 = Aspen, 6 = Douglas-fir, 7 = Krummholz.

## 4. Load Dataset

`scikit-learn` provides this real dataset directly through a built-in fetch function.
We just need to call it once and it will download and load the data into a pandas
DataFrame. (Internet is required the first time — after that it is cached on your
computer. The download is about 75 MB.)

In [ ]:
# Step 1: Import the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_covtype

print("Libraries imported successfully")

In [ ]:
# Step 2: Load the real Forest Cover Type dataset
covtype = fetch_covtype(as_frame=True)
df = covtype.frame

print("Dataset loaded successfully")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Step 3: Look at the first few rows (only key columns, for readability)
key_columns = ["Elevation", "Aspect", "Slope",
               "Horizontal_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways",
               "Horizontal_Distance_To_Fire_Points", "Cover_Type"]
df[key_columns].head()

In [ ]:
# Step 4: Basic information about the dataset
df[key_columns].info()

In [ ]:
# Step 5: Check for missing values
df[key_columns].isnull().sum()

In [ ]:
# Step 6: Give the Cover_Type numbers proper tree-species names, for easy reading
cover_type_names = {
    1: "Spruce/Fir",
    2: "Lodgepole Pine",
    3: "Ponderosa Pine",
    4: "Cottonwood/Willow",
    5: "Aspen",
    6: "Douglas-fir",
    7: "Krummholz"
}
df["Cover_Type_Name"] = df["Cover_Type"].map(cover_type_names)

df[["Cover_Type", "Cover_Type_Name"]].head()

## 5. Dependent and Independent Variables

| Variable | Type | Meaning |
|---|---|---|
| **Elevation** | **Dependent** | **The terrain elevation we want to study (meters)** |
| Aspect | Independent | Compass direction the slope faces |
| Slope | Independent | Steepness of the slope |
| Horizontal_Distance_To_Hydrology | Independent | Distance to nearest water |
| Horizontal_Distance_To_Roadways | Independent | Distance to nearest road |
| Horizontal_Distance_To_Fire_Points | Independent | Distance to nearest fire point |
| Cover_Type_Name | Independent (categorical) | Dominant tree species |

The **dependent variable (Y)** is the one we are trying to explain: `Elevation`.
The **independent variables (X)** are the terrain factors that might affect it.

In [ ]:
# Dependent variable
Y = df["Elevation"]

# Independent variables (using the key numeric ones for simplicity)
independent_columns = ["Aspect", "Slope", "Horizontal_Distance_To_Hydrology",
                        "Horizontal_Distance_To_Roadways",
                        "Horizontal_Distance_To_Fire_Points"]
X = df[independent_columns]

print("Dependent variable   :", "Elevation")
print("Independent variables:", independent_columns)

## 6. Mean vs Median Comparison

- **Mean** = sum of values / number of values (average). Gets affected by very large or
  very small values (outliers).
- **Median** = the middle value when data is sorted. Not affected by outliers.

Rule of thumb:
- Mean ≈ Median → data is symmetric.
- Mean > Median → data is right-skewed (some very high values pull the mean up).
- Mean < Median → data is left-skewed (some very low values pull the mean down).

In [ ]:
# Mean and Median of Elevation
mean_elevation = df["Elevation"].mean()
median_elevation = df["Elevation"].median()

print("Mean of Elevation  :", mean_elevation)
print("Median of Elevation:", median_elevation)

if mean_elevation > median_elevation:
    print("Interpretation: Mean > Median -> data is RIGHT-SKEWED")
elif mean_elevation < median_elevation:
    print("Interpretation: Mean < Median -> data is LEFT-SKEWED")
else:
    print("Interpretation: Mean = Median -> data is SYMMETRIC")

In [ ]:
# Let's also check Mean vs Median for Horizontal_Distance_To_Roadways
mean_road = df["Horizontal_Distance_To_Roadways"].mean()
median_road = df["Horizontal_Distance_To_Roadways"].median()

print("Mean of Horizontal_Distance_To_Roadways  :", mean_road)
print("Median of Horizontal_Distance_To_Roadways:", median_road)

## 7. Variance vs IQR (Interquartile Range)

- **Variance** and **Standard Deviation** measure how spread out the data is from the
  mean. They use every value, so outliers affect them a lot.
- **IQR = Q3 - Q1** measures the spread of the middle 50% of the data. It is not
  affected much by outliers.

A big difference between what standard deviation suggests and what IQR suggests usually
means the data has outliers.

In [ ]:
# Variance and Standard Deviation
variance_elevation = df["Elevation"].var()
std_elevation = df["Elevation"].std()

print("Variance of Elevation          :", variance_elevation)
print("Standard Deviation of Elevation:", std_elevation)

In [ ]:
# IQR calculation
Q1 = df["Elevation"].quantile(0.25)
Q3 = df["Elevation"].quantile(0.75)
IQR = Q3 - Q1

print("Q1 (25th percentile):", Q1)
print("Q3 (75th percentile):", Q3)
print("IQR (Q3 - Q1)       :", IQR)

In [ ]:
# Finding outliers using the 1.5 x IQR rule
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[(df["Elevation"] < lower_limit) | (df["Elevation"] > upper_limit)]

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("Number of outliers found:", len(outliers))

## 8. Skewness and Distribution Shape

**Skewness** tells us if the data is symmetric or leans to one side.

| Skewness value | Meaning |
|---|---|
| close to 0 | Symmetric distribution |
| greater than 0 | Right-skewed (long tail on the right) |
| less than 0 | Left-skewed (long tail on the left) |

In [ ]:
# Skewness of Elevation
skewness_elevation = df["Elevation"].skew()
print("Skewness of Elevation:", skewness_elevation)

if skewness_elevation > 0:
    print("Interpretation: Right-skewed distribution (long tail towards high elevation)")
elif skewness_elevation < 0:
    print("Interpretation: Left-skewed distribution (long tail towards low elevation)")
else:
    print("Interpretation: Symmetric distribution")

In [ ]:
# Skewness of a few more columns
print("Skewness of Slope                              :", df["Slope"].skew())
print("Skewness of Horizontal_Distance_To_Hydrology   :", df["Horizontal_Distance_To_Hydrology"].skew())
print("Skewness of Horizontal_Distance_To_Roadways    :", df["Horizontal_Distance_To_Roadways"].skew())

## 9. Histogram, Boxplot, and Density Plot

- **Histogram** shows how frequently values occur, using bars.
- **Boxplot** shows the median, quartiles, and outliers.
- **Density plot** is a smooth curve version of the histogram.

In [ ]:
# Histogram of Elevation
plt.figure(figsize=(8, 5))
plt.hist(df["Elevation"], bins=50, color="skyblue", edgecolor="black")
plt.title("Histogram of Elevation")
plt.xlabel("Elevation (meters)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Boxplot of Elevation
plt.figure(figsize=(6, 5))
sns.boxplot(y=df["Elevation"], color="lightgreen")
plt.title("Boxplot of Elevation")
plt.ylabel("Elevation (meters)")
plt.show()

In [ ]:
# Density plot of Elevation
plt.figure(figsize=(8, 5))
sns.kdeplot(df["Elevation"], fill=True, color="orange")
plt.title("Density Plot of Elevation")
plt.xlabel("Elevation (meters)")
plt.show()

**Interpretation:** The histogram and density plot both show that elevation is
roughly bell-shaped but with a slight lean, which matches the skewness value calculated
above. The boxplot shows the median elevation and a small number of outlier points,
representing unusually high or low patches of land compared to the rest of the forest.

## 10. Pivot Tables

A pivot table helps us summarize data by groups. Here we will build pivot tables to see
how average elevation and slope change across the 7 forest **cover types**.

In [ ]:
# Step 1: Take a random sample of 20,000 rows to make grouping faster and easier to read
# (the full dataset has over half a million rows)
df_sample = df.sample(n=20000, random_state=1)

df_sample["Cover_Type_Name"].value_counts()

In [ ]:
# Step 2: Pivot table - Average Elevation by Cover Type
pivot1 = pd.pivot_table(df_sample, values="Elevation", index="Cover_Type_Name", aggfunc="mean")
pivot1

In [ ]:
# Step 3: Pivot table - Average Elevation and Slope by Cover Type
pivot2 = pd.pivot_table(df_sample, values=["Elevation", "Slope"],
                         index="Cover_Type_Name", aggfunc="mean")
pivot2

In [ ]:
# Step 4: Visualize average Elevation by Cover Type as a bar chart
plt.figure(figsize=(9, 5))
pivot1["Elevation"].sort_values().plot(kind="barh", color="teal")
plt.title("Average Elevation by Forest Cover Type")
plt.xlabel("Average Elevation (meters)")
plt.show()

**Interpretation:** The pivot table shows that different tree species grow at clearly
different average elevations — for example, **Krummholz** (a stunted, wind-shaped tree)
grows at much higher average elevation than **Cottonwood/Willow**, which prefers
low-lying areas near water. This matches real ecological knowledge and shows how
grouped summaries reveal patterns that a single overall mean/median would hide.

## Lab Summary

In this lab, we used the **real Forest Cover Type dataset** to:
1. State a real-world problem — studying terrain elevation across forest cover types.
2. Plan our analysis approach step by step.
3. Understand the features available in the dataset.
4. Load the dataset using `scikit-learn`'s `fetch_covtype`.
5. Identify dependent and independent variables.
6. Compare mean and median to detect skew.
7. Compare variance and IQR to detect spread and outliers.
8. Compute skewness to describe the shape of the distribution.
9. Visualize the distribution using histogram, boxplot, and density plot.
10. Build pivot tables to summarize data by forest cover type.

### Practice Exercises
1. Find the mean and median of `Slope`. Is it symmetric or skewed?
2. Calculate the IQR of `Horizontal_Distance_To_Fire_Points` and count the outliers.
3. Make a histogram and boxplot of `Horizontal_Distance_To_Hydrology`.
4. Create a pivot table showing average `Horizontal_Distance_To_Roadways` by
   `Cover_Type_Name`.
5. Calculate skewness of `Vertical_Distance_To_Hydrology` and interpret the result.